In [1]:
import pandas as pd
import yfinance as yf
from helpers import drop_nan_days, filter_daily_volume
import pyarrow.parquet as pq

In [15]:
tmp = pd.read_csv("AllUSData-27-02-2026.csv", header=[0, 1], index_col=0)

In [42]:
tmp = pd.read_parquet("AllUSData-2y-03-03-2026-pyarrow.parquet")

In [43]:
tmp

Price        Close                                                  \
Ticker        AACB AACBR  AACBU  AACG AACIU AACOU    AAL      AAME   
Date                                                                 
2024-03-04     NaN   NaN    NaN  1.49   NaN   NaN  14.81  2.576890   
2024-03-05     NaN   NaN    NaN  1.46   NaN   NaN  14.67  2.645217   
2024-03-06     NaN   NaN    NaN  1.61   NaN   NaN  14.88  2.654978   
2024-03-07     NaN   NaN    NaN  1.51   NaN   NaN  14.90  2.733065   
2024-03-08     NaN   NaN    NaN  1.57   NaN   NaN  14.68  2.762348   
...            ...   ...    ...   ...   ...   ...    ...       ...   
2026-02-24  10.341   NaN  10.60  0.93  9.98  9.94  13.15  2.700000   
2026-02-25  10.341   NaN  10.60  1.00  9.97  9.95  13.32  2.690000   
2026-02-26  10.341   NaN  10.60  0.98  9.97  9.95  13.94  2.690000   
2026-02-27  10.341   NaN  10.60  1.02  9.97  9.94  13.07  2.620000   
2026-03-02  10.341   0.3  10.53  0.99  9.97  9.95  12.52  2.590000   

Price                               ... High  Low Open Volume Adj Close Close  \
Ticker            AAOI        AAON  ... VLN+ VLN+ VLN+   VLN+     WFC-Z WFC-Z   
Date                                ...                                         
2024-03-04   14.960000   81.830383  ...  NaN  NaN  NaN    NaN       NaN   NaN   
2024-03-05   14.740000   80.263199  ...  NaN  NaN  NaN    NaN       NaN   NaN   
2024-03-06   14.900000   80.342560  ...  NaN  NaN  NaN    NaN       NaN   NaN   
2024-03-07   14.840000   80.461578  ...  NaN  NaN  NaN    NaN       NaN   NaN   
2024-03-08   14.760000   81.086464  ...  NaN  NaN  NaN    NaN       NaN   NaN   
...                ...         ...  ...  ...  ...  ...    ...       ...   ...   
2026-02-24   56.270000  100.949997  ...  NaN  NaN  NaN    NaN       NaN   NaN   
2026-02-25   58.119999   99.050003  ...  NaN  NaN  NaN    NaN       NaN   NaN   
2026-02-26   53.689999   98.889999  ...  NaN  NaN  NaN    NaN       NaN   NaN   
2026-02-27   84.230003  101.199997  ...  NaN  NaN  NaN    NaN       NaN   NaN   
2026-03-02  102.510002  104.730003  ...  NaN  NaN  NaN    NaN       NaN   NaN   

Price       High   Low  Open Volume  
Ticker     WFC-Z WFC-Z WFC-Z  WFC-Z  
Date                                 
2024-03-04   NaN   NaN   NaN    NaN  
2024-03-05   NaN   NaN   NaN    NaN  
2024-03-06   NaN   NaN   NaN    NaN  
2024-03-07   NaN   NaN   NaN    NaN  
2024-03-08   NaN   NaN   NaN    NaN  
...          ...   ...   ...    ...  
2026-02-24   NaN   NaN   NaN    NaN  
2026-02-25   NaN   NaN   NaN    NaN  
2026-02-26   NaN   NaN   NaN    NaN  
2026-02-27   NaN   NaN   NaN    NaN  
2026-03-02   NaN   NaN   NaN    NaN  

[500 rows x 37252 columns]

In [73]:
df = tmp.copy()

In [74]:
df.index = pd.to_datetime(df.index)

In [ ]:
start_date = '2025-01-01'
end_date = '2025-12-31'
cutted_df = df[(df.index >= start_date) & (df.index <= end_date)]

: 

In [78]:
cutted_df = filter_daily_volume(cutted_df, 2000000, True)
cutted_df = drop_nan_days(cutted_df, "Close", 2)

In [73]:
df = yf.download(["AAPL", "SNDK", "WDC", "LITE", "AXTI"], period="1y", interval="1d")

C:\Users\sasso\AppData\Local\Temp\ipykernel_86472\3897251713.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(["AAPL", "SNDK", "WDC", "LITE", "AXTI"], period="1y", interval="1d")
[*********************100%***********************]  5 of 5 completed


In [ ]:
closes = cutted_df["Close"]

In [80]:
change_df = closes.resample('ME').apply(lambda x: ((x.iloc[-1] - x.iloc[0]) / x.iloc[0]) * 100).to_period('M')

In [81]:
ema_returns = change_df.ewm(span=6).mean()
final_scores = ema_returns.iloc[-1]
final_scores.sort_values(ascending=False).head(20)

Ticker
CAPR    90.794111
RGC     61.923299
PRAX    61.873943
ABVX    60.084137
SBET    50.522775
OLMA    50.441532
SLS     44.365085
PACS    43.014908
OMER    41.725347
TERN    40.253177
GPCR    39.930057
WVE     39.135151
COGT    38.554505
TRON    32.185425
ONDS    31.777723
CELC    31.683664
KOD     31.447121
PL      31.371123
TE      31.180712
FEIM    29.139847
Name: 2025-12, dtype: float64

In [82]:
change_df_t = change_df.transpose()
change_df_t["FinalScore"] = final_scores
change_df_t.sort_values("FinalScore", ascending=False).head(15)

Date,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,FinalScore
Ticker,,,,,,,,,,,,,
CAPR,0.534759,1.317522,-30.577910,33.966249,-22.370483,-2.742407,-15.670098,-20.506327,13.902056,-16.751917,-11.861619,333.984965,90.794111
RGC,-21.161764,20.735469,686.967771,106.967797,1038.960947,-13.431990,-9.589045,8.326907,20.233458,5.180036,-14.970506,75.732220,61.923299
PRAX,-3.734902,-51.906308,12.876303,4.934487,1.287435,5.600200,25.596487,-15.806987,17.935028,274.877391,7.950987,59.698740,61.873943
ABVX,-18.106998,29.292925,-14.733969,20.882853,-12.879883,20.472444,836.945145,12.627700,3.549218,24.296112,26.282250,12.454134,60.084137
SBET,-35.820897,-13.953485,-32.558137,-18.181819,2104.022889,-82.066100,101.176457,3.967330,0.176682,-20.322398,-18.869368,-6.875008,50.522775
OLMA,9.694793,-26.677855,-10.688837,54.491017,-0.189031,-2.628566,17.620137,2.439027,51.313760,-18.749997,236.342036,-10.554564,50.441532
SLS,51.401862,-20.270275,-6.896545,42.201826,5.555558,38.607595,-23.557688,15.662650,-12.021859,7.692307,-9.497205,151.333332,44.365085
PACS,12.374319,-9.071879,-13.935686,-11.244242,2.057611,31.702340,-14.594590,9.604524,18.464188,-13.898787,183.375742,14.528636,43.014908
OMER,-12.398377,-0.237253,0.611997,-7.079645,-55.982907,-2.597400,30.303025,15.235456,-4.428906,77.053144,35.146436,77.479336,41.725347


In [ ]:
full_df_change_t = full_df_change.transpose()
full_df_change_t["FinalScore"] = final_scores
full_df_change_t.sort_values("FinalScore", ascending=False).head(15)

Date,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,...,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,FinalScore
Ticker,,,,,,,,,,,,,,,,,,,,,
PPCB,10.344826,-56.521738,137.500010,-27.777776,-38.461540,-42.857142,-40.000000,0.000000,0.000000,-33.333338,...,140.722881,-68.541668,-37.728936,-27.058825,-20.920355,-43.066313,-60.153926,-15.683703,0.0,3194.358878
VSME,-10.952382,13.573405,-20.823244,-10.714286,-29.901962,-29.629632,-2.934784,32.530126,-1.666675,9.243688,...,33.971295,66.417908,0.799999,-71.270493,-73.584906,-30.821914,1763.157898,-30.978263,0.0,240.946451
SMX,-33.846154,11.278195,13.907284,-20.754717,-50.243902,-40.085289,7.407412,-86.535715,-38.904902,207.291672,...,-66.823530,-77.142856,-13.095243,-87.234721,3348.587642,-58.066173,-33.510891,-42.745681,0.0,220.159223
ANL,9.617275,46.808519,-48.619956,-46.905536,7.361964,-15.959659,-26.752768,0.000000,9.852219,15.732759,...,-4.666670,16.551724,1.067419,-20.555556,14.388493,-3.401365,670.289866,-15.210359,0.0,94.279335
BNAI,-37.845303,-61.394556,157.819908,-26.439235,-35.802475,-35.359118,-10.344834,-14.300003,-1.204824,29.866664,...,-16.873456,-9.144548,-2.649004,35.562313,-15.268818,-35.376045,556.498677,53.001719,0.0,87.890176
BVC,-4.477616,-40.000001,-56.505574,-45.022286,-23.684210,77.474389,-36.206894,334.999996,-26.766921,29.746836,...,-21.222221,1.141229,100.719424,-15.671644,16.018516,-3.389827,527.142849,-21.540177,0.0,76.653844
ELPW,0.281423,0.277775,0.276750,1.011956,0.180991,0.225527,1.079136,-33.985763,-81.955924,-6.382973,...,339.675175,-91.126761,-1.618122,-1.219515,-27.000004,-46.344337,626.041661,-91.940298,0.0,70.406434
XWEL,-18.357483,6.586827,-8.771935,12.499995,16.477271,-2.564106,-6.989247,3.468205,-20.348838,7.857144,...,10.869561,9.900993,-10.000001,-11.111112,-14.606741,-41.025638,-28.888889,380.645161,0.0,67.474720
CDIO,10.077513,-50.719427,-0.716331,-21.316163,-16.666671,-18.300656,-32.424239,35.265694,16.923073,206.688957,...,14.681440,-1.804128,3.084836,-5.164325,-27.638188,-6.228376,-46.830985,351.724136,0.0,61.759757


In [156]:
change_t = change_df.transpose()
change_t["FinalScore"] = final_scores
change_t.sort_values("FinalScore", ascending=False).head(15)

Date,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,FinalScore
Ticker,,,,,,,,
SBET,2.777775,-32.558137,-18.181819,2104.022889,-82.066100,101.176457,3.967330,262.065955
ABVX,-0.130042,-14.733969,20.882853,-12.879883,20.472444,836.945145,12.627700,187.390194
RGC,-5.349697,686.967771,106.967797,1038.960947,-13.431990,-9.589045,8.326907,181.129090
ASST,-3.623185,15.169655,13.076929,1088.135548,-46.581194,-10.962573,103.642379,154.460000
ZEPP,-2.197800,10.447760,-15.584417,-6.111110,12.711873,377.304977,246.153846,153.159881
DFDV,0.260499,19.019302,1389.799396,48.966943,8.777278,-31.556402,14.021354,132.787533
NAKA,-10.638300,-5.294119,18.238991,1102.127683,-34.969471,-46.950772,-14.330706,114.933589
VOR,-2.702704,-24.579837,9.516132,-69.742815,592.307750,44.755248,-6.190473,94.580859
NEGG,0.000000,-29.729733,-28.846150,34.444452,203.551403,306.305743,-21.670089,92.585130
